# CIG-Bench · RGT demo

Run the pretrained relative geological time (RGT) model on a 3D seismic
volume and visualize the result.

This notebook contains **no `cigvis` calls** — every figure is plain matplotlib.
For each section we show three panels:

1. seismic in grayscale,
2. the predicted RGT volume in `jet`,
3. the seismic with horizon iso-contours of the RGT overlaid (jet-colored,
   so a given color denotes the same relative geological time across panels).

Sections are sampled evenly along the inline (axis 0) and xline (axis 1)
directions.  Time slices are skipped because RGT iso-contours degenerate to
constant values on a time slice.

本 notebook 不使用 cigvis；对每条剖面显示三栏 —— 灰度地震 / `jet` 配色的
RGT 体 / 在地震上叠加 RGT 等值线（颜色按 RGT 值映射到 `jet`，因此同一颜色
在所有面上代表同一相对地质时间）。沿 inline (axis 0) 和 xline (axis 1) 两
个方向各均匀抽 4 条剖面；time 方向被跳过，因为时间切片上 RGT 几乎为常值。


## 1. Imports

In [ ]:
import numpy as np
import torch

from cig_bench.predictor.rgt import RGTPredictor

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)


## 2. Plot helper — three-panel RGT sections

`plot_rgt_sections(seis, rgt, axis=0|1, ...)` draws a column of three-panel
rows along the chosen axis.  Contour levels are picked uniformly between
`rgt.min()` and `rgt.max()` and colored with the same `jet` colormap as the
RGT panel, so iso-contour colors are comparable across all sections.

`plot_rgt_sections(seis, rgt, axis=0|1, ...)` 沿所选方向均匀抽剖面，每行
三栏。等值线水平在 `rgt.min()/max()` 之间均匀分布，并使用与 RGT 面相同的
`jet` 配色，因此等值线颜色在所有剖面上代表同一 RGT 值。


In [ ]:
import numpy as np
import matplotlib.pyplot as plt


def plot_rgt_sections(seis, rgt, axis, n_sec=4, n_levels=45,
                      figsize_per_row=3.2, suptitle=None):
    """Three-panel sections — seismic (gray) | RGT (jet) | seismic + contours.

    Parameters
    ----------
    seis, rgt : (T, H, W) numpy arrays.
    axis      : 0 = inline (along H), 1 = xline (along W).
    n_sec     : number of evenly-spaced sections (skips the two edges).
    n_levels  : how many RGT iso-contours to draw on the overlay panel.
    """
    T, H, W = seis.shape
    if axis == 0:
        idx = np.linspace(0, H - 1, n_sec + 2, dtype=int)[1:-1]
        get = lambda v, i: v[:, i, :]
        ax_name = "inline"; x_lab, y_lab = "crossline", "depth"
    elif axis == 1:
        idx = np.linspace(0, W - 1, n_sec + 2, dtype=int)[1:-1]
        get = lambda v, i: v[:, :, i]
        ax_name = "xline"; x_lab, y_lab = "inline", "depth"
    else:
        raise ValueError("axis must be 0 or 1 for RGT contour overlay")

    levels = np.linspace(rgt.min() + 0.02, rgt.max() - 0.02, n_levels)

    fig, axes = plt.subplots(n_sec, 3, figsize=(13, figsize_per_row * n_sec))
    if n_sec == 1:
        axes = axes[None, :]

    for row, i in enumerate(idx):
        s = get(seis, i); r = get(rgt, i)
        vmax = np.percentile(np.abs(s), 99)
        ax_s, ax_r, ax_o = axes[row]

        ax_s.imshow(s, cmap="gray", aspect="auto", vmin=-vmax, vmax=vmax)
        ax_s.set_title(f"{ax_name} {i} — seismic", fontsize=10)

        ax_r.imshow(r, cmap="jet", aspect="auto",
                    vmin=float(rgt.min()), vmax=float(rgt.max()))
        ax_r.set_title(f"{ax_name} {i} — RGT", fontsize=10)

        ax_o.imshow(s, cmap="gray", aspect="auto", vmin=-vmax, vmax=vmax)
        ax_o.contour(r, levels=levels, cmap="jet",
                     vmin=float(rgt.min()), vmax=float(rgt.max()),
                     linewidths=1.2)
        ax_o.set_title(f"{ax_name} {i} — seismic + horizons", fontsize=10)

        for ax in axes[row]:
            ax.set_xlabel(x_lab); ax.set_ylabel(y_lab)

    if suptitle:
        fig.suptitle(suptitle, fontsize=13, y=1.0)
    fig.tight_layout()
    plt.show()


## 3. Load seismic

The volume layout convention is `(T, H, W)`.

地震体的形状约定是 `(T, H, W)`。


In [ ]:
seis = np.load(r"../RealData/your_seis.npy").astype(np.float32)   # (T, H, W)
print("seis shape (T, H, W):", seis.shape)


## 4. Build the predictor and run inference

`RGTPredictor()` loads the RGT-Est-style HRNet weights from ModelScope on
first use.  The two optional auxiliary input channels (`horizon_rgt`,
`horizon_mask`) are kept as `None` here, which is equivalent to "no horizon
constraints".

> ⚠️ The inference grid is fixed to `(400, 512, 512)` by the predictor's
> default `infer_shape`; the model resamples to/from it automatically.  Pass
> a different `infer_shape=` to `RGTPredictor(...)` only if you know your
> checkpoint was trained on a different size.

> ⚠️ 推理网格默认是 `(400, 512, 512)`，predictor 内部会自动重采样到该尺寸
> 再变回原始尺寸；只有当你的 checkpoint 是按其它尺寸训练的，才需要给
> `RGTPredictor(...)` 传不同的 `infer_shape=`。


In [ ]:
predictor = RGTPredictor(device=device)              # auto-downloads weights
rgt_volume, seis_used = predictor.predict(seis)

print("rgt shape  :", rgt_volume.shape,
      "min/max:", float(rgt_volume.min()), float(rgt_volume.max()))
print("seis shape :", seis_used.shape)


## 5. Section visualisations

In [ ]:
plot_rgt_sections(seis_used, rgt_volume, axis=0, n_sec=4,
                  suptitle="RGT — inline sections")


In [ ]:
plot_rgt_sections(seis_used, rgt_volume, axis=1, n_sec=4,
                  suptitle="RGT — xline sections")
